In [ ]:
import sys
sys.path.append('..')
sys.path.append("../College/DataPrep/")
sys.path.append("../College/Model/")

import importlib
from College.DataPrep import Prep_Map, Output_Map, Player_Dataset, Data_Prep
importlib.reload(Data_Prep)
importlib.reload(Prep_Map)
importlib.reload(Output_Map)
importlib.reload(Player_Dataset)
from College.DataPrep.Data_Prep import College_Data_Prep, College_IO
from College.DataPrep.Player_Dataset import Create_Test_Train_Datasets
from Constants import device

In [ ]:
batch_size = 4000
num_epochs = 50

In [ ]:
import torch
torch.set_printoptions(precision=8, sci_mode=False)
torch.set_printoptions(
    precision=2,
    sci_mode=False,
    linewidth=500,
    edgeitems=20,
)

In [ ]:
data_prep = College_Data_Prep(Prep_Map.college_base_prep_map, Output_Map.college_output_map)
pitcher_io_list = data_prep.Generate_IO_Pitchers("WHERE LastYear<=? AND isPitcher=?", (2019, 1), use_cutoff=True)
train_dataset, test_dataset = Create_Test_Train_Datasets(pitcher_io_list, 0.25, 0)

In [ ]:
from College.Model import College_Model, Model_Train
importlib.reload(College_Model)
importlib.reload(Model_Train)
from torch.optim import lr_scheduler
from College.Model.College_Model import RNN_Model
from College.Model.Model_Train import trainAndGraph

network = RNN_Model(train_dataset.get_input_size(), 
                    data_prep=data_prep, 
                    is_hitter=False)
network = network.to(device)

optimizer = torch.optim.Adam(network.parameters(), lr=0.0025)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5, cooldown=5, verbose=False)

best_losses = trainAndGraph(network,
              train_dataset,
              test_dataset,
              scheduler,
              optimizer,
              batch_size,
              num_epochs = num_epochs,
              logging_interval=10,
              early_stopping_cutoff=2000,
              should_output=True,
              model_name="../Models/default_college_pitcher",
              save_last=True,
              elements_to_save=[0])

print(best_losses)

In [ ]:
import torch.nn.functional as F
from Constants import _DRAFT_BUCKETS_LIST as DBL

def ViewPlayerPrediction(id : int, net, players : list[College_IO]):
    with torch.no_grad():
        net.eval()
        for player in players:
            if player.player.TBCId != id:
                continue
            
            data = player.input.to(device).unsqueeze(0)
            length = torch.tensor([player.length]).to(device)
            
            output_draft = net(data, length)
            output_draft = output_draft.squeeze(0)
            output_draft = F.softmax(output_draft, dim=-1)
            
            for i in range(output_draft.size(0)):
                s = f"Year {i + 1}: "
                for j in range(1, len(DBL) - 1):
                    s += f"{DBL[j-1] + 1}-{DBL[j]}: {output_draft[i,j + 1]:.3f}, "
                s += f"{DBL[-1]}+ : {output_draft[i,6]:.3f}, Undrafted : {output_draft[i,0]:.3f}"
                print(s)
                
            print(f"Draft Bucket: {player.output_draft[0].item()}")
            
            # Show if player was or wasn't in training
            for p in train_dataset.bio:
                if p.TBCId == id:
                    print("In Training Data")
                    break
                
            for p in test_dataset.bio:
                if p.TBCId == id:
                    print("In Testing Data")
                    break

In [ ]:
ViewPlayerPrediction(196290, network, pitcher_io_list)

In [ ]:
ViewPlayerPrediction(193237, network, pitcher_io_list)

In [ ]:
ViewPlayerPrediction(187338, network, pitcher_io_list)

In [ ]:
ViewPlayerPrediction(193582, network, pitcher_io_list)